In [1]:
import sys

sys.path.append(r"C:\Users\david\PycharmProjects\normalizing-flows")
sys.path.append(r"C:\Users\david\PycharmProjects\nfmc")
sys.path.append(r"C:\Users\david\PycharmProjects\mcmc-diagnostics")
sys.path.append(r"C:\Users\david\PycharmProjects\potentials")
sys.path.append(r"C:\Users\david\PycharmProjects\potentials")

In [2]:
import torch
from nfmc.util import sum_except_batch


class DiagonalGaussian:
    """
    Class for the diagonal Gaussian distribution.
    """

    def __init__(self, event_shape, mu: float = 3.0, std: float = 2.0):
        self.event_shape = event_shape
        self.mu = mu
        self.std = std

    @property
    def first_moment(self):
        return torch.full(size=self.event_shape, fill_value=self.mu)

    @property
    def variance(self):
        return torch.full(size=self.event_shape, fill_value=self.std**2)

    @property
    def second_moment(self):
        return self.variance + self.first_moment**2

    def neg_log_prob(self, x: torch.Tensor):
        """
        Computes the negative log probability density of this distribution.

        :param torch.Tensor x: input tensor with shape `(*batch_shape, *event_shape)`.
        :return: negative log probability density tensor with shape `batch_shape`.
        """
        return sum_except_batch(
            (x - self.mu) ** 2 / (2 * self.std**2), self.event_shape
        )


In [3]:
from torchflows import Flow, RealNVP

torch.manual_seed(0)

event_shape = (4,)
n_chains = 4
target = DiagonalGaussian(event_shape)
flow = Flow(RealNVP(event_shape))

In [4]:
from nfmc.algorithms.preconditioning.implementations import NeuTraRWMHKernel
from nfmc.algorithms.preconditioning.base import PreconditionedMCMCSampler

kernel = NeuTraRWMHKernel(flow=flow, neg_log_prob_target=target.neg_log_prob)
sampler = PreconditionedMCMCSampler(kernel)

z0 = torch.rand(size=(n_chains, *event_shape)) * 2 - 1
warmup_draws, latent_warmup_draws = sampler.warmup(
    z0=z0,
    n_steps=1200,
    preconditioner_update_interval=500,
    return_latent_samples=True,
)
sampling_draws = sampler.sample(z0=latent_warmup_draws.last_sample, n_steps=2000)

Sampling (Preconditioned RWMH): 100%|██████████| 2000/2000 [00:09<00:00, 211.60it/s, Preconditioned RWMH, 0.0 c/s, 0.0 g/s]
